# Robustness — multiple hidden mapping-class elements

Public reproducibility version for the manuscript *Mapping-Class-Group Learning for Diffeomorphisms*.

**Important:** the revised manuscript reports candidate selection by unpenalized validation loss (equivalently, `lambda = 0`). The original exploratory notebooks also logged a reference word-length penalty of `1e-7`; that legacy setting did not change the selected candidate in the reported runs.

Run Jupyter from the repository root so that relative output paths under `results/` resolve as intended.



# Step 4: Multiple hidden mapping-class words

This notebook extends the original \(G_\star=AB\) proof-of-concept experiment to several hidden mapping-class elements of different word lengths.

The four **new** hidden words are
\[
{\rm id},\qquad A,\qquad ABA,\qquad ABAB,
\]
with lengths \(0,1,3,4\), respectively. Together with the original \(AB\) experiment, this gives coverage of hidden-word lengths \(0,1,2,3,4\).

The candidate set is enlarged from the original 9 candidates to 15 candidates:
\[
\{
{\rm id},
A,A^{-1},B,B^{-1},
AB,BA,A^2,B^2,
ABA,BAB,A^2B,AB^2,ABAB,A^2B^2
\}.
\]

The experiment uses the same PNN architecture and training budget as the original uniform-sampling experiment:
- \(N_{\rm train}=2000\), \(N_{\rm val}=1000\), \(N_{\rm test}=1000\);
- uniform sampling on \(T^2\);
- PNN architecture \(4\to128\to128\to2\);
- \(\tanh\) activations;
- residual scale \(0.1\);
- Adam, learning rate \(10^{-3}\);
- full-batch training;
- 3000 epochs per candidate;
- no early stopping;
- \(\lambda=10^{-7}\);
- five repetitions with seeds \(0,1,2,3,4\).

For every hidden word and seed, the notebook reports:
1. the selected candidate;
2. whether the target candidate has the smallest **unpenalized** validation loss;
3. the validation margin
   \[
   \widehat\gamma =
   \min_{G\neq G_\star}L_{\rm val}(G)
   -L_{\rm val}(G_\star);
   \]
4. the final test loss;
5. the mean and standard deviation across five seeds.


In [ ]:

import os
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

QUICK_TEST = False

# Repository-relative output directory.
os.makedirs("results/multiple_hidden_words", exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]
N_TRAIN = 2000
N_VAL   = 1000
N_TEST  = 1000

EPOCHS = 3000
LR = 1e-3
HIDDEN_WIDTH = 128
RESIDUAL_SCALE = 0.1
# Revised manuscript: primary selector is unpenalized validation loss.
# The original 1e-7 value is retained only in archived/raw results.
LAMBDA_WORD = 0.0
if QUICK_TEST:
    SEEDS = [0]
    N_TRAIN = 200
    N_VAL = 100
    N_TEST = 100
    EPOCHS = 20

device = torch.device("cpu")

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not enforce deterministic algorithms:", e)

print("device:", device)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("QUICK_TEST:", QUICK_TEST)


In [ ]:

def torus_diff(pred, target):
    return (pred - target + 0.5) % 1.0 - 0.5

def torus_mse(pred, target):
    return torch.mean(torus_diff(pred, target) ** 2)


In [ ]:

A = np.array([[1, 1],
              [0, 1]], dtype=np.int64)

A_INV = np.array([[ 1, -1],
                  [ 0,  1]], dtype=np.int64)

B = np.array([[1, 0],
              [1, 1]], dtype=np.int64)

B_INV = np.array([[ 1,  0],
                  [-1,  1]], dtype=np.int64)

TOKEN_MATRIX = {
    "A": A,
    "Ainv": A_INV,
    "B": B,
    "Binv": B_INV,
}

def word_matrix(tokens):
    M = np.eye(2, dtype=np.int64)
    for token in tokens:
        M = M @ TOKEN_MATRIX[token]
    return M

def matrix_map(M):
    Mt = torch.tensor(M.T, dtype=torch.float32, device=device)
    def G(x):
        return (x @ Mt) % 1.0
    return G


In [ ]:

candidate_tokens = {
    "id":      [],
    "A":       ["A"],
    "A^-1":    ["Ainv"],
    "B":       ["B"],
    "B^-1":    ["Binv"],
    "AB":      ["A", "B"],
    "BA":      ["B", "A"],
    "A^2":     ["A", "A"],
    "B^2":     ["B", "B"],
    "ABA":     ["A", "B", "A"],
    "BAB":     ["B", "A", "B"],
    "A^2B":    ["A", "A", "B"],
    "AB^2":    ["A", "B", "B"],
    "ABAB":    ["A", "B", "A", "B"],
    "A^2B^2":  ["A", "A", "B", "B"],
}

candidates = {}

for name, tokens in candidate_tokens.items():
    M = word_matrix(tokens)
    candidates[name] = {
        "tokens": tokens,
        "matrix": M,
        "length": len(tokens),
        "map": matrix_map(M),
    }

seen = {}
for name, info in candidates.items():
    key = tuple(info["matrix"].reshape(-1).tolist())
    if key in seen:
        raise ValueError(f"Equivalent duplicate candidates: {seen[key]} and {name}")
    seen[key] = name

candidate_table = pd.DataFrame(
    [
        {
            "candidate": name,
            "length": info["length"],
            "matrix": str(info["matrix"].tolist()),
            "max_abs_matrix_entry": int(np.abs(info["matrix"]).max()),
        }
        for name, info in candidates.items()
    ]
)

display(candidate_table)
print("Number of distinct candidates:", len(candidates))


In [ ]:

HIDDEN_WORDS = ["id", "A", "ABA", "ABAB"]

hidden_table = candidate_table[
    candidate_table["candidate"].isin(HIDDEN_WORDS)
].copy()

hidden_table["order"] = hidden_table["candidate"].map(
    {name: i for i, name in enumerate(HIDDEN_WORDS)}
)
hidden_table = hidden_table.sort_values("order").drop(columns="order")

display(hidden_table)

for name in HIDDEN_WORDS:
    print(f"{name:5s}  length={candidates[name]['length']}")
    print(candidates[name]["matrix"])
    print()


In [ ]:

def F0(x, eps=0.05):
    u = (
        x[..., 0]
        + eps * torch.sin(2 * torch.pi * (x[..., 0] - x[..., 1]))
    ) % 1.0

    v = (
        x[..., 1]
        + eps * torch.sin(2 * torch.pi * x[..., 1])
    ) % 1.0

    return torch.stack([u, v], dim=-1)

def make_target(hidden_name, eps=0.05):
    G_star = candidates[hidden_name]["map"]
    def target(x):
        return F0(G_star(x), eps=eps)
    return target


In [ ]:

class PeriodicResidualNN(torch.nn.Module):
    def __init__(self, hidden=128, residual_scale=0.1):
        super().__init__()
        self.residual_scale = residual_scale

        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, 2),
        )

    def periodic_features(self, x):
        return torch.cat(
            [
                torch.cos(2 * torch.pi * x),
                torch.sin(2 * torch.pi * x),
            ],
            dim=-1,
        )

    def forward(self, x):
        features = self.periodic_features(x)
        dx = self.residual_scale * self.net(features)
        return (x + dx) % 1.0


In [ ]:

def train_one_candidate(
    candidate_name,
    x_train,
    y_train,
    x_val,
    y_val,
    x_test,
    y_test,
    seed,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = PeriodicResidualNN(
        hidden=HIDDEN_WIDTH,
        residual_scale=RESIDUAL_SCALE,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    G = candidates[candidate_name]["map"]

    with torch.no_grad():
        gx_train = G(x_train)
        gx_val = G(x_val)
        gx_test = G(x_test)

    model.train()
    for _ in range(EPOCHS):
        optimizer.zero_grad()
        pred = model(gx_train)
        loss = torus_mse(pred, y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        train_loss = torus_mse(model(gx_train), y_train).item()
        val_loss = torus_mse(model(gx_val), y_val).item()
        test_loss = torus_mse(model(gx_test), y_test).item()

    word_length = candidates[candidate_name]["length"]
    penalized_score = val_loss + LAMBDA_WORD * word_length

    return {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "test_loss": test_loss,
        "word_length": word_length,
        "penalized_score": penalized_score,
    }


In [ ]:

all_rows = []
run_summaries = []

total_trainings = len(SEEDS) * len(HIDDEN_WORDS) * len(candidates)
completed_trainings = 0
global_start = time.time()

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    x_train = torch.rand(N_TRAIN, 2, device=device)
    x_val   = torch.rand(N_VAL, 2, device=device)
    x_test  = torch.rand(N_TEST, 2, device=device)

    for hidden_name in HIDDEN_WORDS:
        target = make_target(hidden_name, eps=0.05)

        with torch.no_grad():
            y_train = target(x_train)
            y_val   = target(x_val)
            y_test  = target(x_test)

        print("=" * 72)
        print(f"seed={seed} | hidden={hidden_name}")
        print("=" * 72)

        rows_this_run = []

        for candidate_name in candidates:
            t0 = time.time()

            result = train_one_candidate(
                candidate_name,
                x_train, y_train,
                x_val, y_val,
                x_test, y_test,
                seed=seed,
            )

            row = {
                "seed": seed,
                "hidden_word": hidden_name,
                "hidden_length": candidates[hidden_name]["length"],
                "candidate": candidate_name,
                **result,
            }

            all_rows.append(row)
            rows_this_run.append(row)

            completed_trainings += 1

            elapsed = time.time() - global_start
            avg_per_training = elapsed / completed_trainings
            remaining = total_trainings - completed_trainings
            eta_min = avg_per_training * remaining / 60.0

            print(
                f"{candidate_name:8s} "
                f"val={result['val_loss']:.4e} "
                f"score={result['penalized_score']:.4e} "
                f"({time.time()-t0:.1f}s; ETA ~{eta_min:.1f} min)"
            )

        run_df = pd.DataFrame(rows_this_run)

        target_row = run_df[run_df["candidate"] == hidden_name].iloc[0]
        wrong_df = run_df[run_df["candidate"] != hidden_name]

        best_wrong_idx = wrong_df["val_loss"].idxmin()
        best_wrong_row = wrong_df.loc[best_wrong_idx]

        margin = best_wrong_row["val_loss"] - target_row["val_loss"]

        selected_penalized = run_df.loc[
            run_df["penalized_score"].idxmin(), "candidate"
        ]

        selected_unpenalized = run_df.loc[
            run_df["val_loss"].idxmin(), "candidate"
        ]

        selected_test_loss = run_df.loc[
            run_df["penalized_score"].idxmin(), "test_loss"
        ]

        run_summary = {
            "seed": seed,
            "hidden_word": hidden_name,
            "hidden_length": candidates[hidden_name]["length"],
            "selected_penalized": selected_penalized,
            "selected_unpenalized": selected_unpenalized,
            "correct_penalized": selected_penalized == hidden_name,
            "correct_unpenalized": selected_unpenalized == hidden_name,
            "target_val_loss": target_row["val_loss"],
            "best_wrong_candidate": best_wrong_row["candidate"],
            "best_wrong_val_loss": best_wrong_row["val_loss"],
            "validation_margin": margin,
            "selected_test_loss": selected_test_loss,
        }

        run_summaries.append(run_summary)

        print()
        print("Run summary:")
        print(run_summary)

        pd.DataFrame(all_rows).to_csv(
            "results/multiple_hidden_words/multiple_hidden_words_all_candidate_results.csv",
            index=False,
        )
        pd.DataFrame(run_summaries).to_csv(
            "results/multiple_hidden_words/multiple_hidden_words_run_summaries.csv",
            index=False,
        )

print("\nFull experiment completed.")


In [ ]:

run_summary_df = pd.DataFrame(run_summaries)

display(
    run_summary_df.sort_values(
        ["hidden_length", "hidden_word", "seed"]
    )
)

print("\nSelection counts (lambda = 0):")
display(
    pd.crosstab(
        run_summary_df["hidden_word"],
        run_summary_df["selected_penalized"],
    )
)


In [ ]:

def sample_sd(x):
    x = np.asarray(x, dtype=float)
    return np.std(x, ddof=1) if len(x) > 1 else np.nan

summary_rows = []

for hidden_name in HIDDEN_WORDS:
    d = run_summary_df[
        run_summary_df["hidden_word"] == hidden_name
    ]

    summary_rows.append({
        "hidden_word": hidden_name,
        "length": candidates[hidden_name]["length"],
        "matrix": str(candidates[hidden_name]["matrix"].tolist()),
        "max_abs_matrix_entry": int(np.abs(candidates[hidden_name]["matrix"]).max()),
        "correct_penalized_count": int(d["correct_penalized"].sum()),
        "n_seeds": len(d),
        "selection_rate": float(d["correct_penalized"].mean()),
        "target_val_mean": float(d["target_val_loss"].mean()),
        "target_val_sd": sample_sd(d["target_val_loss"]),
        "margin_mean": float(d["validation_margin"].mean()),
        "margin_sd": sample_sd(d["validation_margin"]),
        "test_loss_mean": float(d["selected_test_loss"].mean()),
        "test_loss_sd": sample_sd(d["selected_test_loss"]),
    })

summary_df = pd.DataFrame(summary_rows)

display(summary_df)

summary_df.to_csv(
    "results/multiple_hidden_words/multiple_hidden_words_aggregate_summary.csv",
    index=False,
)


In [ ]:

paper_rows = []

for _, r in summary_df.iterrows():
    paper_rows.append({
        "Hidden word": r["hidden_word"],
        "Length": int(r["length"]),
        "Max |matrix entry|": int(r["max_abs_matrix_entry"]),
        "Selected correctly": f"{int(r['correct_penalized_count'])}/{int(r['n_seeds'])}",
        "Val. loss (mean ± SD)": f"{r['target_val_mean']:.3e} ± {r['target_val_sd']:.3e}",
        "Margin (mean ± SD)": f"{r['margin_mean']:.3e} ± {r['margin_sd']:.3e}",
        "Test loss (mean ± SD)": f"{r['test_loss_mean']:.3e} ± {r['test_loss_sd']:.3e}",
    })

paper_table_df = pd.DataFrame(paper_rows)

display(paper_table_df)

paper_table_df.to_csv(
    "results/multiple_hidden_words/multiple_hidden_words_paper_table.csv",
    index=False,
)


In [ ]:

plot_df = summary_df.sort_values("length")

plt.figure(figsize=(7, 5))
plt.errorbar(
    plot_df["length"],
    plot_df["margin_mean"],
    yerr=plot_df["margin_sd"],
    marker="o",
    capsize=4,
)
plt.axhline(0, linewidth=1)
plt.xlabel("Hidden-word length")
plt.ylabel("Validation margin")
plt.title("Multiple hidden words: mean validation margin over five seeds")
plt.grid(True)
plt.show()



## What to send back after the run

The most useful outputs are:

1. the `paper_table_df` table;
2. the `run_summary_df` table if any seed fails;
3. the CSV file `multiple_hidden_words_aggregate_summary.csv`.

The main manuscript result will be especially clean if all four new hidden words are selected \(5/5\) times and the mean validation margins remain positive.
